# Laboratório 5 — Câmera Estéreo (versão base — a completar)

**Disciplina:** ESZA019 — Visão Computacional — UFABC 2026.2
**Professor:** Celso Setsuo Kurashima
**Equipe:** Ctrl+C, Ctrl+V e Fé

### Integrantes da equipe
- Lucas Rodrigues Teixeira — RA 11202131394
- Pedro Henrique Garcez Silva — RA 11202130642
- Roberto Sene Azevedo — RA 11202020360

**Data de realização dos experimentos:** _a preencher_
**Data de publicação do relatório:** _a preencher_

> **Aviso — versão base.** Este notebook contém a **estrutura completa e a fundamentação teórica** do
> Laboratório 5, além dos **programas já elaborados** pela equipe (`capture_images_abc.py`,
> `calibrate_abc.py`). Os **resultados experimentais** (fotos da câmera estéreo construída, pares de
> calibração, parâmetros obtidos, vídeo 3D anaglifo e percepções da equipe) **ainda não foram coletados**
> e estão marcados ao longo do texto como **[A COMPLETAR PELA EQUIPE]**. Basta preencher esses pontos e
> inserir as evidências para finalizar o relatório.


## Sumário

1. [Introdução](#1-introdução)
2. [Fundamentação teórica (Parte 1)](#2-fundamentação-teórica-parte-1)
   - 2.1 [Estereoscopia e visão 3D](#21-estereoscopia-e-visão-3d)
   - 2.2 [Geometria epipolar: epipolos, plano e linha epipolar](#22-geometria-epipolar-epipolos-plano-e-linha-epipolar)
   - 2.3 [Matriz fundamental e matriz essencial](#23-matriz-fundamental-e-matriz-essencial)
   - 2.4 [Disparidade estéreo](#24-disparidade-estéreo)
   - 2.5 [Calibração estéreo e retificação](#25-calibração-estéreo-e-retificação)
3. [Construção da câmera estéreo (Parte 2)](#3-construção-da-câmera-estéreo-parte-2)
4. [Ambiente e instruções de reprodução](#4-ambiente-e-instruções-de-reprodução)
5. [Procedimentos experimentais (Parte 3)](#5-procedimentos-experimentais-parte-3)
   - 5.1 [(A) Obtenção dos códigos de exemplo](#51-a-obtenção-dos-códigos-de-exemplo)
   - 5.2 [(B) Execução do exemplo com as imagens fornecidas](#52-b-execução-do-exemplo-com-as-imagens-fornecidas)
   - 5.3 [(C) Calibração da câmera estéreo construída](#53-c-calibração-da-câmera-estéreo-construída)
   - 5.4 [(D) Gravação de um vídeo 3D anaglifo](#54-d-gravação-de-um-vídeo-3d-anaglifo)
   - 5.5 [(E) Aplicação no Trabalho Final](#55-e-aplicação-no-trabalho-final)
6. [Análise e discussão](#6-análise-e-discussão)
7. [Declaração de uso de Inteligência Artificial Generativa](#7-declaração-de-uso-de-inteligência-artificial-generativa)
8. [Referências](#8-referências)


## 1. Introdução

Nos laboratórios anteriores a equipe estudou a captura de imagens (Lab 1), *features* e correspondência
(Lab 2), homografia e mosaico (Lab 3) e a **calibração de uma única câmera** (Lab 4). Neste Laboratório
5 damos o passo natural seguinte: usar **duas câmeras** para recuperar a **profundidade** de uma cena —
a **visão estéreo**.

O objetivo é entender a **geometria epipolar** que relaciona duas vistas de uma mesma cena, **construir**
fisicamente uma câmera estereoscópica de baixo custo com duas webcams iguais, **calibrá-la** e, a partir
das imagens retificadas, gerar uma **imagem/vídeo 3D anaglifo** (para óculos vermelho-ciano). Este
relatório reúne a teoria necessária, o procedimento de construção e os programas de captura e calibração,
deixando indicados os pontos em que os resultados experimentais da equipe devem ser inseridos.


## 2. Fundamentação teórica (Parte 1)

> Esta seção também responde às perguntas teóricas da Parte 1 do roteiro (epipolos, plano epipolar, linha
> epipolar, parâmetros da matriz fundamental e disparidade estéreo).


### 2.1 Estereoscopia e visão 3D

A **estereoscopia** é o princípio pelo qual dois pontos de vista ligeiramente diferentes de uma mesma
cena permitem recuperar a **profundidade** — exatamente como os dois olhos humanos. Cada olho (ou câmera)
vê o mundo de uma posição distinta; o cérebro (ou o algoritmo) compara as duas imagens e, a partir do
**deslocamento** dos objetos entre elas, infere a que distância cada um está. Objetos próximos deslocam-se
muito entre as duas vistas; objetos distantes, pouco.

A distância entre os dois centros ópticos chama-se **baseline**. Para imitar a visão humana, o roteiro
pede que a baseline seja aproximadamente igual à **distância interpupilar** de um integrante da equipe.


### 2.2 Geometria epipolar: epipolos, plano e linha epipolar

A **geometria epipolar** é a geometria projetiva intrínseca a **duas vistas**. Considere um ponto 3D $X$
observado pelas duas câmeras, com centros ópticos $O_L$ e $O_R$:

- **Plano epipolar:** o plano que contém o ponto $X$ e os dois centros ópticos $O_L$ e $O_R$. Todo ponto
  da cena define um plano epipolar (todos passam pela reta $O_L O_R$, a *baseline*).
- **Epipolos ($e_L$, $e_R$):** o ponto onde a reta que liga os dois centros ópticos (a *baseline*)
  fura cada plano de imagem. Equivalentemente, o epipolo de uma câmera é a **imagem do centro óptico da
  outra câmera**. Todas as linhas epipolares de uma imagem passam pelo seu epipolo.
- **Linha epipolar:** a interseção do plano epipolar com o plano de imagem. Sua importância é prática: o
  correspondente de um ponto $x_L$ da imagem esquerda **necessariamente** está sobre a linha epipolar
  correspondente na imagem direita (a **restrição epipolar**). Isso reduz a busca de correspondência de
  um problema 2D para um problema **1D** ao longo de uma reta.


### 2.3 Matriz fundamental e matriz essencial

A restrição epipolar é codificada por uma matriz $3\times3$. Para pontos em coordenadas de **pixel**
$x_L$ e $x_R$, vale

$$x_R^{\top}\, F\, x_L = 0,$$

onde $F$ é a **matriz fundamental**. Suas características:

- É $3\times3$ e tem **posto 2** (é singular, $\det F = 0$).
- É definida **a menos de escala**, o que lhe dá **7 graus de liberdade** (9 elementos $-$ 1 de escala
  $-$ 1 da restrição de posto). Por isso pode ser estimada a partir de **correspondências de pontos** com
  o algoritmo dos 8 pontos (`cv2.findFundamentalMat`).
- Ela mapeia um ponto de uma imagem na sua **linha epipolar** na outra: $l_R = F\,x_L$.

Quando as câmeras estão **calibradas** (conhecemos os intrínsecos $K_L, K_R$), usa-se a **matriz
essencial** $E = K_R^{\top} F\,K_L$, que opera sobre coordenadas normalizadas e se decompõe diretamente
na **rotação $R$** e **translação $t$** entre as duas câmeras — os extrínsecos do par estéreo.


### 2.4 Disparidade estéreo

Depois de **retificar** o par (ver 2.5), as linhas epipolares ficam horizontais e um mesmo ponto do mundo
aparece na **mesma linha** nas duas imagens, deslocado apenas horizontalmente. Esse deslocamento é a
**disparidade**:

$$d = x_L - x_R.$$

A disparidade relaciona-se com a profundidade $Z$ por

$$Z = \frac{f \cdot B}{d},$$

onde $f$ é a distância focal (em pixels) e $B$ a *baseline*. A relação é **inversa**: disparidade grande
= objeto **próximo**; disparidade pequena = objeto **distante**; disparidade nula = ponto no infinito. O
**mapa de disparidade** (calculado, por exemplo, com `cv2.StereoBM`/`StereoSGBM`) é a base para o mapa de
profundidade explorado no laboratório seguinte.


### 2.5 Calibração estéreo e retificação

Calibrar o par estéreo envolve três etapas, todas presentes no programa `calibrate_abc.py` da equipe:

1. **Calibração individual (mono)** de cada câmera (`cv2.calibrateCamera`), obtendo $K_L, dist_L$ e
   $K_R, dist_R$ — exatamente como no Lab 4.
2. **Calibração estéreo** (`cv2.stereoCalibrate` com `CALIB_FIX_INTRINSIC`), que, mantendo os intrínsecos,
   estima a **rotação $R$** e a **translação $T$** entre as câmeras, além das matrizes **Essencial $E$** e
   **Fundamental $F$**. O módulo $\lVert T \rVert$ é a *baseline* em milímetros.
3. **Retificação** (`cv2.stereoRectify` + `cv2.initUndistortRectifyMap`), que gera os mapas que deixam as
   linhas epipolares **horizontais e alinhadas**, além da matriz de reprojeção **$Q$** (usada para levar
   disparidade a coordenadas 3D). Todos esses parâmetros são salvos em um arquivo **`.xml`**.


## 3. Construção da câmera estéreo (Parte 2)

Conforme a referência [2] (*Making A Low-Cost Stereo Camera Using OpenCV*), a equipe deve montar uma
câmera estéreo com **duas webcams USB iguais**, fixadas **paralelamente** sobre uma base rígida (caixa de
sapato, placa, etc.), com os eixos ópticos paralelos e uma *baseline* próxima à **distância interpupilar**
de um integrante. Após a montagem, as câmeras **não podem mais se mover** uma em relação à outra (a
calibração deixa de valer se isso ocorrer).

**[A COMPLETAR PELA EQUIPE]**
- Foto(s) da câmera estéreo construída (e, se houver, o nome/decoração dados a ela).
- Distância interpupilar medida: **____ mm** (integrante: ____), usada como *baseline* alvo.
- Descrição dos materiais e do método de fixação, com detalhes suficientes para reprodução.


## 4. Ambiente e instruções de reprodução

Ambiente **Linux**, **Python 3** e **OpenCV** (o `requirements.txt` da pasta fixa
`opencv-contrib-python>=4.8`, `numpy>=1.24`, `tqdm>=4.65`). O padrão de calibração é o **mesmo tabuleiro
do Lab 4**, com **cantos internos $(8, 6)$**.

```bash
# 1) Ambiente virtual + dependencias
python3 -m venv venv && source venv/bin/activate
pip install -r requirements.txt

# 2) Capturar de 10 a 15 pares (L/R) do tabuleiro pelas duas webcams
python3 capture_images_abc.py

# 3) Calibrar a camera estereo (gera data/params_py.xml)
python3 calibrate_abc.py

# 4) Exibir ao vivo em 3D anaglifo / gravar video 3D (programas a elaborar — item D)
python3 movie3d_abc.py
python3 movie3d_abc_gravacao.py
```

> **Atenção aos índices das câmeras.** No Linux as webcams aparecem como `/dev/video*`. Muitas webcams
> criam **dois** nós por dispositivo, por isso no programa de captura a câmera direita usa o índice `2`.
> Ajuste `CamL_id` / `CamR_id` conforme o resultado de `v4l2-ctl --list-devices`.


## 5. Procedimentos experimentais (Parte 3)

### 5.1 (A) Obtenção dos códigos de exemplo

**Objetivo:** baixar os códigos de exemplo da câmera estéreo do repositório da LearnOpenCV
(`stereo-camera`, referência [2]), que contém três programas Python (`calibrate.py`,
`capture_images.py` e o de exibição 3D) e as subpastas de imagens **L** (esquerda) e **R** (direita).

Os arquivos de exemplo `calibrate.py`, `capture_images.py` (e as versões em C++ `calibrate.cpp`,
`capture_images.cpp`) já estão presentes nesta pasta e serviram de base para os programas adaptados da
equipe. **Observação do roteiro:** ao final da aula no laboratório, os arquivos baixados no computador do
laboratório devem ser apagados.


### 5.2 (B) Execução do exemplo com as imagens fornecidas

**Objetivo:** executar `calibrate.py` (calibração estéreo) e o programa de imagem 3D com as imagens
**fornecidas** no exemplo, para validar o pipeline antes de usar a câmera própria.

**Parâmetros necessários para uma câmera estéreo** (resposta ao roteiro): para cada câmera, os
**intrínsecos** $K_L, K_R$ e os coeficientes de **distorção** $dist_L, dist_R$; e, para o par, a
**rotação $R$** e a **translação $T$** entre as câmeras, as matrizes **Essencial $E$** e **Fundamental
$F$**, os mapas de **retificação** e a matriz de reprojeção **$Q$**. **Há arquivo necessário?** Sim: a
calibração produz um arquivo de parâmetros (`.xml`) que precisa ser carregado pelo programa de exibição
3D — sem ele não é possível retificar as imagens ao vivo.

**[A COMPLETAR PELA EQUIPE]** Saída da execução de `calibrate.py` com as imagens de exemplo (parâmetros
e erro de reprojeção) e um quadro da imagem 3D gerada.


### 5.3 (C) Calibração da câmera estéreo construída

**Objetivo:** capturar de 10 a 15 **pares** de imagens do tabuleiro com a câmera estéreo da equipe
(`capture_images_abc.py`) e calibrá-la (`calibrate_abc.py`), listando todos os parâmetros obtidos.

O programa de captura abaixo (elaborado pela equipe) abre as **duas** webcams com o backend V4L2, mostra
os quadros lado a lado, detecta o tabuleiro em tempo real como *feedback* visual e salva o par
**apenas quando o tabuleiro é visto pelas duas câmeras ao mesmo tempo**.


In [ ]:
#!/usr/bin/env python3
# capture_images_abc.py — captura pares (L/R) do tabuleiro com DUAS webcams USB.
import os
import cv2

CamL_id = 0            # webcam da ESQUERDA (/dev/video0)
CamR_id = 2            # webcam da DIREITA  (muitas webcams criam 2 nós; por isso 2)
NOME_EQUIPE = "roberto"
CHESSBOARD = (8, 6)    # CANTOS INTERNOS do tabuleiro (nao quadrados)
FRAME_W, FRAME_H = 640, 480
pathL, pathR = "./data/stereoL/", "./data/stereoR/"

def abrir_camera(cam_id, w, h):
    """Abre a webcam USB via backend V4L2, fixando resolucao e MJPG (mais FPS)."""
    cap = cv2.VideoCapture(cam_id, cv2.CAP_V4L2)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, w)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, h)
    cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
    return cap

def main():
    os.makedirs(pathL, exist_ok=True)
    os.makedirs(pathR, exist_ok=True)
    CamL = abrir_camera(CamL_id, FRAME_W, FRAME_H)
    CamR = abrir_camera(CamR_id, FRAME_W, FRAME_H)
    if not CamL.isOpened() or not CamR.isOpened():
        print("ERRO: nao consegui abrir uma das cameras. Cheque CamL_id/CamR_id e o USB.")
        return

    print("Cameras abertas. 'l' = salvar par (se o tabuleiro estiver nas DUAS) | q = sair")
    contador = 1
    while True:
        retL, frameL = CamL.read()
        retR, frameR = CamR.read()
        if not (retL and retR):
            continue

        grayL = cv2.cvtColor(frameL, cv2.COLOR_BGR2GRAY)   # deteccao trabalha em cinza
        grayR = cv2.cvtColor(frameR, cv2.COLOR_BGR2GRAY)
        okL, cornersL = cv2.findChessboardCorners(grayL, CHESSBOARD, None)
        okR, cornersR = cv2.findChessboardCorners(grayR, CHESSBOARD, None)

        visL, visR = frameL.copy(), frameR.copy()          # feedback visual
        if okL: cv2.drawChessboardCorners(visL, CHESSBOARD, cornersL, okL)
        if okR: cv2.drawChessboardCorners(visR, CHESSBOARD, cornersR, okR)
        status = "OK: pode salvar" if (okL and okR) else "ajuste o tabuleiro..."
        cor = (0, 255, 0) if (okL and okR) else (0, 0, 255)
        cv2.putText(visL, f"{status}  salvos={contador-1}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, cor, 2)
        cv2.imshow("ESQUERDA | DIREITA", cv2.hconcat([visL, visR]))

        tecla = cv2.waitKey(1) & 0xFF
        if tecla == ord("q"):
            break
        elif tecla == ord("l") and okL and okR:            # so salva com tabuleiro nas DUAS
            cv2.imwrite(pathL + "lucas%d.png" % contador, frameL)
            cv2.imwrite(pathR + "lucas%d.png" % contador, frameR)
            print(f"par {contador} salvo."); contador += 1

    CamL.release(); CamR.release(); cv2.destroyAllWindows()
    print(f"Fim. Total de pares salvos: {contador-1}")

if __name__ == "__main__":
    main()


O programa de calibração `calibrate_abc.py` (também da equipe) lê automaticamente todos os pares
salvos, detecta e refina os cantos, calibra cada câmera individualmente, executa a **calibração estéreo**
e a **retificação**, e salva **todos** os parâmetros em `data/params_py.xml` (intrínsecos, `dist`, $R$,
$T$, $E$, $F$, $Q$ e a *baseline*), pois o Lab 6 precisará deles. Trechos essenciais:


In [ ]:
# calibrate_abc.py (trechos) — calibracao estereo a partir dos pares L/R.
import glob, numpy as np, cv2

CHESSBOARD = (8, 6)
SQUARE_SIZE_MM = 25.0      # tamanho REAL do quadrado -> da escala metrica (medir com regua!)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# Pontos 3D do padrao (Z=0), em milimetros.
objp = np.zeros((CHESSBOARD[0]*CHESSBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD[0], 0:CHESSBOARD[1]].T.reshape(-1, 2)
objp *= SQUARE_SIZE_MM

# ... (deteccao dos cantos em cada par L/R, com cornerSubPix) ...

# 1) Calibracao MONO de cada camera:
retL, mtxL, distL, *_ = cv2.calibrateCamera(obj_pts, img_ptsL, image_size, None, None)
retR, mtxR, distR, *_ = cv2.calibrateCamera(obj_pts, img_ptsR, image_size, None, None)
new_mtxL, _ = cv2.getOptimalNewCameraMatrix(mtxL, distL, image_size, 1, image_size)
new_mtxR, _ = cv2.getOptimalNewCameraMatrix(mtxR, distR, image_size, 1, image_size)

# 2) Calibracao ESTEREO (mantendo intrinsecos): estima R, T, E, F entre as cameras.
flags = cv2.CALIB_FIX_INTRINSIC
retS, new_mtxL, distL, new_mtxR, distR, Rot, Trns, Emat, Fmat = cv2.stereoCalibrate(
    obj_pts, img_ptsL, img_ptsR, new_mtxL, distL, new_mtxR, distR,
    image_size, criteria, flags)
baseline_mm = float(np.linalg.norm(Trns))   # baseline = |T|

# 3) RETIFICACAO: linhas epipolares horizontais + matriz de reprojecao Q.
rect_l, rect_r, proj_l, proj_r, Q, roiL, roiR = cv2.stereoRectify(
    new_mtxL, distL, new_mtxR, distR, image_size, Rot, Trns, 1, (0, 0))

# 4) Salva TUDO em data/params_py.xml (mtxL/R, distL/R, R, T, E, F, Q, baseline, mapas de retificacao).


**[A COMPLETAR PELA EQUIPE]** Depois de capturar os pares e rodar `calibrate_abc.py`, inserir aqui:
- $K_L$, $dist_L$, $K_R$, $dist_R$ (intrínsecos e distorção de cada câmera);
- $R$, $T$ entre as câmeras e a **baseline** $\lVert T \rVert$ obtida (comparar com a interpupilar medida);
- matrizes $E$, $F$ e $Q$; e o **erro de reprojeção estéreo (RMS)**;
- **quais parâmetros foram salvos no arquivo `.xml`** (resposta ao roteiro: `mtxL/R`, `distL/R`, `R`,
  `T`, `E`, `F`, `Q`, `baseline_mm`, os mapas de retificação e o tamanho da imagem).


### 5.4 (D) Gravação de um vídeo 3D anaglifo

**Objetivo:** a partir do programa de exibição ao vivo (`movie3d_abc.py`), elaborar
`movie3d_abc_gravacao.py` para **gravar** um vídeo 3D **anaglifo** (canais vermelho/ciano, para óculos 3D)
de aproximadamente **10 a 20 s**, e convertê-lo para **mp4**.

A ideia do anaglifo é combinar as duas vistas **retificadas** em uma só imagem: o **canal vermelho** vem
de uma câmera e os **canais verde+azul (ciano)** da outra; com os óculos vermelho-ciano, cada olho recebe
a vista correspondente e o cérebro reconstrói a profundidade. Esboço do laço de gravação:


In [ ]:
# movie3d_abc_gravacao.py (esboço a validar pela equipe)
import cv2, numpy as np

# 1) Carregar parametros salvos e os mapas de retificacao.
fs = cv2.FileStorage("data/params_py.xml", cv2.FILE_STORAGE_READ)
Lx = fs.getNode("Left_Stereo_Map_x").mat();  Ly = fs.getNode("Left_Stereo_Map_y").mat()
Rx = fs.getNode("Right_Stereo_Map_x").mat(); Ry = fs.getNode("Right_Stereo_Map_y").mat()
fs.release()

capL = cv2.VideoCapture(0, cv2.CAP_V4L2)
capR = cv2.VideoCapture(2, cv2.CAP_V4L2)

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('anaglifo.avi', fourcc, 20.0, (640, 480))   # ~20 fps

while True:
    okL, fL = capL.read(); okR, fR = capR.read()
    if not (okL and okR):
        break
    # 2) Retificar cada quadro com os mapas (linhas epipolares horizontais).
    rL = cv2.remap(fL, Lx, Ly, cv2.INTER_LINEAR)
    rR = cv2.remap(fR, Rx, Ry, cv2.INTER_LINEAR)
    # 3) Compor o anaglifo: R da esquerda, G+B da direita.
    anaglifo = np.zeros_like(rL)
    anaglifo[:, :, 2] = rL[:, :, 2]          # canal vermelho  (olho esquerdo)
    anaglifo[:, :, 0] = rR[:, :, 0]          # canal azul      (olho direito)
    anaglifo[:, :, 1] = rR[:, :, 1]          # canal verde     (olho direito)
    out.write(anaglifo)
    cv2.imshow("Anaglifo 3D (oculos vermelho-ciano)", anaglifo)
    if cv2.waitKey(1) == ord('q'):
        break

capL.release(); capR.release(); out.release(); cv2.destroyAllWindows()
# Converter para mp4:  ffmpeg -i anaglifo.avi anaglifo.mp4


**[A COMPLETAR PELA EQUIPE]**
- Criar de fato o `movie3d_abc.py` / `movie3d_abc_gravacao.py` (a partir do exemplo da referência [2]) e
  validar o esboço acima com os parâmetros reais.
- Gravar o vídeo 3D (~10–20 s), convertê-lo para **mp4** e incorporá-lo/linká-lo aqui.
- Registrar a **percepção individual de cada integrante** ao assistir com os óculos anaglifo e
  **comparar** a experiência ao vivo com a do vídeo gravado.


### 5.5 (E) Aplicação da câmera estéreo no Trabalho Final

**Objetivo:** discutir, com argumentos técnicos e de aplicação real, como a câmera estéreo construída pode
ser usada no Trabalho Final da disciplina.

A câmera estéreo entrega **profundidade métrica** da cena, o que habilita aplicações como: **estimativa de
distância a obstáculos** (navegação de robôs móveis e veículos), **reconstrução 3D** de objetos e
ambientes, **medição de dimensões** reais, **detecção de pessoas/objetos com posição no espaço** e
**realidade aumentada** com oclusão correta. O fio condutor é sempre a matriz $Q$ obtida na calibração,
que converte disparidade em coordenadas 3D.

**[A COMPLETAR PELA EQUIPE]** Texto próprio relacionando a câmera estéreo ao tema específico do Trabalho
Final da equipe, ilustrado com desenhos/figuras.


## 6. Análise e discussão

**[A COMPLETAR PELA EQUIPE]** Discutir, à luz dos resultados obtidos: a qualidade da calibração estéreo
(erro de reprojeção), a coerência entre a **baseline** estimada ($\lVert T \rVert$) e a distância
interpupilar medida, o efeito da **retificação** (linhas epipolares horizontais), a qualidade da
percepção 3D no anaglifo e as principais dificuldades (alinhamento/fixação das webcams, iluminação,
sincronismo das câmeras) e soluções adotadas.


## 7. Declaração de uso de Inteligência Artificial Generativa

Em atendimento à **Portaria CNPq nº 2664/2026**, a equipe declara o uso de assistente de IA generativa
(modelo de linguagem da Anthropic — Claude) como **ferramenta de apoio** na preparação desta base de
relatório.

- **Finalidade:** organização e estruturação das seções, apoio à redação da fundamentação teórica
  (geometria epipolar, matriz fundamental, disparidade) e comentários do código já elaborado pela equipe.
- **Autoria e responsabilidade:** a concepção do trabalho, as decisões técnicas, os programas e os
  **dados experimentais** (construção da câmera, calibração, gravação 3D e percepções) são **de autoria da
  equipe**. A ferramenta de IA **não gera dados nem resultados experimentais**; a equipe revisa e valida
  todo o conteúdo e é **integralmente responsável** pelo material final.


## 8. Referências

1. LearnOpenCV — *Introduction to Epipolar Geometry and Stereo Vision*. Disponível em:
   https://learnopencv.com/introduction-to-epipolar-geometry-and-stereo-vision/
2. LearnOpenCV — *Making A Low-Cost Stereo Camera Using OpenCV*. Disponível em:
   https://learnopencv.com/making-a-low-cost-stereo-camera-using-opencv/
   (código: https://github.com/spmallick/learnopencv/tree/master/stereo-camera)
3. LearnOpenCV — *Understanding Lens Distortion*. Disponível em:
   https://learnopencv.com/understanding-lens-distortion/
4. LOOP, C.; ZHANG, Z. *Computing Rectifying Homographies for Stereo Vision*. IEEE CVPR, 1999.
5. LearnOpenCV — *Geometry of Image Formation*. Disponível em:
   https://learnopencv.com/geometry-of-image-formation/
6. CNPq — *Portaria nº 2664/2026* (diretrizes de integridade e uso de IAG). Disponível em:
   http://memoria2.cnpq.br/web/guest/view/-/journal_content/56_INSTANCE_0oED/10157/23142775
